In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

base = '/content/drive/MyDrive/rice-pest-cnn/dataset'
classes = [
    'asiatic_rice_borer',
    'paddy_stem_maggot',
    'rice_leaf_caterpillar',
    'rice_leaf_roller',
    'yellow_rice_borer',
    'brown_plant_hopper',
    'rice_leaf_hopper',
    'thrips'
]

for c in classes:
    count = len(os.listdir(os.path.join(base, c)))
    print(f"{c}: {count} images")

asiatic_rice_borer: 745 images
paddy_stem_maggot: 325 images
rice_leaf_caterpillar: 475 images
rice_leaf_roller: 605 images
yellow_rice_borer: 455 images
brown_plant_hopper: 290 images
rice_leaf_hopper: 686 images
thrips: 580 images


In [3]:
import os, shutil, random
from pathlib import Path

# Install library
import subprocess
subprocess.run(['pip', 'install', 'Pillow', '-q'])

# Paths
base = '/content/drive/MyDrive/rice-pest-cnn/dataset'
output = '/content/rice_pest_data'
classes = ['asiatic_rice_borer', 'paddy_stem_maggot', 'rice_leaf_caterpillar', 'rice_leaf_roller', 'yellow_rice_borer', 'brown_plant_hopper', 'rice_leaf_hopper', 'thrips']

# Split ratios: 70% train, 15% val, 15% test
splits = {'train': 0.70, 'val': 0.15, 'test': 0.15}

random.seed(42)

for cls in classes:
    images = os.listdir(os.path.join(base, cls))
    images = [f for f in images if f.endswith('.jpg')]
    random.shuffle(images)

    total = len(images)
    n_train = int(total * 0.70)
    n_val   = int(total * 0.15)

    split_data = {
        'train': images[:n_train],
        'val':   images[n_train:n_train + n_val],
        'test':  images[n_train + n_val:]
    }

    for split, files in split_data.items():
        dest = os.path.join(output, split, cls)
        os.makedirs(dest, exist_ok=True)
        for f in files:
            shutil.copy(
                os.path.join(base, cls, f),
                os.path.join(dest, f)
            )

# Verify
print("Split complete!\n")
for split in ['train', 'val', 'test']:
    print(f"--- {split} ---")
    for cls in classes:
        count = len(os.listdir(os.path.join(output, split, cls)))
        print(f"  {cls}: {count}")

Split complete!

--- train ---
  asiatic_rice_borer: 700
  paddy_stem_maggot: 700
  rice_leaf_caterpillar: 700
  rice_leaf_roller: 700
  yellow_rice_borer: 700
  brown_plant_hopper: 700
  rice_leaf_hopper: 700
  thrips: 700
--- val ---
  asiatic_rice_borer: 111
  paddy_stem_maggot: 48
  rice_leaf_caterpillar: 71
  rice_leaf_roller: 90
  yellow_rice_borer: 68
  brown_plant_hopper: 43
  rice_leaf_hopper: 102
  thrips: 87
--- test ---
  asiatic_rice_borer: 113
  paddy_stem_maggot: 50
  rice_leaf_caterpillar: 72
  rice_leaf_roller: 92
  yellow_rice_borer: 69
  brown_plant_hopper: 44
  rice_leaf_hopper: 104
  thrips: 87


In [4]:
from PIL import Image, ImageEnhance, ImageOps
import random, os
from pathlib import Path

random.seed(42)

def augment_image(img):
    """Apply a random combination of augmentations to one image"""
    # Random horizontal flip
    if random.random() > 0.5:
        img = ImageOps.mirror(img)
    # Random rotation -25 to +25 degrees
    angle = random.uniform(-25, 25)
    img = img.rotate(angle, expand=False)
    # Random brightness 0.7 to 1.3
    img = ImageEnhance.Brightness(img).enhance(random.uniform(0.7, 1.3))
    # Random contrast 0.8 to 1.2
    img = ImageEnhance.Contrast(img).enhance(random.uniform(0.8, 1.2))
    return img

TARGET = 700  # target images per class in train set
train_dir = '/content/rice_pest_data/train'
classes = ['asiatic_rice_borer', 'paddy_stem_maggot', 'rice_leaf_caterpillar', 'rice_leaf_roller', 'yellow_rice_borer', 'brown_plant_hopper', 'rice_leaf_hopper', 'thrips']

for cls in classes:
    cls_path = os.path.join(train_dir, cls)
    images = [f for f in os.listdir(cls_path) if f.endswith('.jpg')]
    current = len(images)

    if current >= TARGET:
        print(f"{cls}: {current} images — no augmentation needed")
        continue

    needed = TARGET - current
    print(f"{cls}: {current} images — generating {needed} more...")

    generated = 0
    while generated < needed:
        # Pick a random source image
        src_name = random.choice(images)
        src_path = os.path.join(cls_path, src_name)
        img = Image.open(src_path).convert('RGB')

        # Augment it
        aug_img = augment_image(img)

        # Save with a new name
        new_name = f"aug_{generated:04d}.jpg"
        aug_img.save(os.path.join(cls_path, new_name), quality=90)
        generated += 1

    print(f"  Done — now has {TARGET} images")

# Final count
print("\nFinal train counts:")
for cls in classes:
    count = len(os.listdir(os.path.join(train_dir, cls)))
    print(f"  {cls}: {count}")

asiatic_rice_borer: 700 images — no augmentation needed
paddy_stem_maggot: 700 images — no augmentation needed
rice_leaf_caterpillar: 700 images — no augmentation needed
rice_leaf_roller: 700 images — no augmentation needed
yellow_rice_borer: 700 images — no augmentation needed
brown_plant_hopper: 700 images — no augmentation needed
rice_leaf_hopper: 700 images — no augmentation needed
thrips: 700 images — no augmentation needed

Final train counts:
  asiatic_rice_borer: 700
  paddy_stem_maggot: 700
  rice_leaf_caterpillar: 700
  rice_leaf_roller: 700
  yellow_rice_borer: 700
  brown_plant_hopper: 700
  rice_leaf_hopper: 700
  thrips: 700


In [5]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# Check GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Image transforms
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Load datasets
data_dir = '/content/rice_pest_data'

train_dataset = datasets.ImageFolder(os.path.join(data_dir, 'train'), transform=train_transforms)
val_dataset   = datasets.ImageFolder(os.path.join(data_dir, 'val'),   transform=val_transforms)
test_dataset  = datasets.ImageFolder(os.path.join(data_dir, 'test'),  transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2)

print(f"\nClass mapping: {train_dataset.class_to_idx}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")

# Load pretrained MobileNetV3
model = models.mobilenet_v3_small(weights='IMAGENET1K_V1')

# Replace the final classifier for 4 classes
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 8)

model = model.to(device)
print(f"\nModel ready on {device}")

Using device: cuda

Class mapping: {'asiatic_rice_borer': 0, 'brown_plant_hopper': 1, 'paddy_stem_maggot': 2, 'rice_leaf_caterpillar': 3, 'rice_leaf_hopper': 4, 'rice_leaf_roller': 5, 'thrips': 6, 'yellow_rice_borer': 7}
Train batches: 175
Val batches:   20

Model ready on cuda


In [6]:
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = StepLR(optimizer, step_size=7, gamma=0.1)

best_val_acc = 0.0
best_model_path = '/content/best_model.pth'

print("Starting training...\n")

for epoch in range(20):
    # --- Training phase ---
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()

    # --- Validation phase ---
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    train_acc = 100. * train_correct / train_total
    val_acc   = 100. * val_correct   / val_total

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        saved = "✓ saved"
    else:
        saved = ""

    scheduler.step()

    print(f"Epoch {epoch+1:2d}/20 | "
          f"Train loss: {train_loss/len(train_loader):.3f} | "
          f"Train acc: {train_acc:.1f}% | "
          f"Val acc: {val_acc:.1f}% {saved}")

print(f"\nBest validation accuracy: {best_val_acc:.1f}%")
print(f"Best model saved to: {best_model_path}")

Starting training...

Epoch  1/20 | Train loss: 0.503 | Train acc: 82.5% | Val acc: 90.6% ✓ saved
Epoch  2/20 | Train loss: 0.161 | Train acc: 94.2% | Val acc: 89.8% 
Epoch  3/20 | Train loss: 0.162 | Train acc: 94.5% | Val acc: 91.8% ✓ saved
Epoch  4/20 | Train loss: 0.093 | Train acc: 96.8% | Val acc: 95.5% ✓ saved
Epoch  5/20 | Train loss: 0.103 | Train acc: 96.6% | Val acc: 94.4% 
Epoch  6/20 | Train loss: 0.087 | Train acc: 97.1% | Val acc: 96.8% ✓ saved
Epoch  7/20 | Train loss: 0.059 | Train acc: 97.9% | Val acc: 95.5% 
Epoch  8/20 | Train loss: 0.030 | Train acc: 98.9% | Val acc: 97.6% ✓ saved
Epoch  9/20 | Train loss: 0.021 | Train acc: 99.2% | Val acc: 98.1% ✓ saved
Epoch 10/20 | Train loss: 0.017 | Train acc: 99.5% | Val acc: 98.2% ✓ saved
Epoch 11/20 | Train loss: 0.017 | Train acc: 99.3% | Val acc: 98.5% ✓ saved
Epoch 12/20 | Train loss: 0.019 | Train acc: 99.4% | Val acc: 98.4% 
Epoch 13/20 | Train loss: 0.012 | Train acc: 99.6% | Val acc: 98.7% ✓ saved
Epoch 14/20 | Trai

In [7]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Load best model
model.load_state_dict(torch.load(best_model_path))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# ✅ Order must match class_to_idx (alphabetical by default in ImageFolder)
class_names = [
    'asiatic_rice_borer',    # 0
    'brown_plant_hopper',    # 1
    'paddy_stem_maggot',     # 2
    'rice_leaf_caterpillar', # 3
    'rice_leaf_hopper',      # 4
    'rice_leaf_roller',      # 5
    'thrips',                # 6
    'yellow_rice_borer'      # 7
]

print("=" * 60)
print("FINAL TEST RESULTS")
print("=" * 60)
print(classification_report(all_labels, all_preds,
                            target_names=class_names, digits=4))

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

FINAL TEST RESULTS
                       precision    recall  f1-score   support

   asiatic_rice_borer     0.9906    0.9292    0.9589       113
    paddy_stem_maggot     0.9565    1.0000    0.9778        44
rice_leaf_caterpillar     0.9412    0.9600    0.9505        50
     rice_leaf_roller     1.0000    1.0000    1.0000        72
    yellow_rice_borer     1.0000    1.0000    1.0000       104
   brown_plant_hopper     1.0000    1.0000    1.0000        92
     rice_leaf_hopper     1.0000    1.0000    1.0000        87
               thrips     0.9315    0.9855    0.9577        69

             accuracy                         0.9826       631
            macro avg     0.9775    0.9843    0.9806       631
         weighted avg     0.9831    0.9826    0.9825       631

Confusion Matrix:
[[105   0   3   0   0   0   0   5]
 [  0  44   0   0   0   0   0   0]
 [  0   2  48   0   0   0   0   0]
 [  0   0   0  72   0   0   0   0]
 [  0   0   0   0 104   0   0   0]
 [  0   0   0   0   0  92   0

In [8]:
import torch.onnx
import shutil
import os # Added import for os module

# Install onnxscript if not already installed
try:
    import onnxscript
except ImportError:
    !pip install onnxscript -q
    import onnxscript

# Save the .pth model to Google Drive
drive_model_dir = '/content/drive/MyDrive/rice-pest-cnn/model'
os.makedirs(drive_model_dir, exist_ok=True)

# Copy best model to Drive
shutil.copy(best_model_path, os.path.join(drive_model_dir, 'best_model.pth'))
print("✓ best_model.pth saved to Google Drive")

# Export to ONNX
model.eval()
dummy_input = torch.randn(1, 3, 224, 224).to(device)
onnx_path = '/content/rice_pest_model.onnx'

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=11,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input':  {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)
print("✓ rice_pest_model.onnx exported")

# Copy ONNX to Drive
shutil.copy(onnx_path, os.path.join(drive_model_dir, 'rice_pest_model.onnx'))
print("✓ rice_pest_model.onnx saved to Google Drive")
print(f"\nBoth files saved to: {drive_model_dir}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 18.2 MB/s eta 0:00:00


/tmp/ipykernel_29886/2612932744.py:25: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0507 22:50:12.960000 29886 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


✓ best_model.pth saved to Google Drive


W0507 22:50:13.961000 29886 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0507 22:50:13.962000 29886 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0507 22:50:13.965000 29886 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0507 22:50:13.966000 29886 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /github/workspace/onnx/version_converter/adapters/axes_input_to_attribute.h:68: adapt: Asserti

✓ rice_pest_model.onnx exported
✓ rice_pest_model.onnx saved to Google Drive

Both files saved to: /content/drive/MyDrive/rice-pest-cnn/model


In [9]:
!pip install onnxscript -q

In [10]:
import onnx

# Load the split model and re-save it as a single file
model_proto = onnx.load("/content/rice_pest_model.onnx")
onnx.save_model(
    model_proto,
    "/content/rice_pest_model_fixed.onnx",
    save_as_external_data=False  # single file, no .data companion
)

# Copy to Google Drive
import shutil
shutil.copy(
    "/content/rice_pest_model_fixed.onnx",
    "/content/drive/MyDrive/rice-pest-cnn/model/rice_pest_model.onnx"
)
print("✓ Fixed ONNX saved to Google Drive")

# Check the file size — should be several MB now
import os
size = os.path.getsize("/content/rice_pest_model_fixed.onnx")
print(f"File size: {size / 1024 / 1024:.1f} MB")


✓ Fixed ONNX saved to Google Drive
File size: 6.1 MB
